# Caso Práctico 3 - Procesamiento de Lenguaje Natural (NLP)
## Tarea: Pregunta-Respuesta utilizando Transformers

## Cargar librerías 

In [ ]:
from transformers import pipeline, TrainingArguments, Trainer, AutoModelForQuestionAnswering, AutoTokenizer
from datasets import load_dataset
import evaluate
import torch

## Prueba inicial del pipeline de Pregunta-Respuesta con un ejemplo controlado

In [4]:
# Usar el modelo BERT base uncased (minúsculas y sin tildes)
qa_model = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", tokenizer="distilbert-base-cased-distilled-squad")

# Definir un contexto de prueba
contexto = "Madrid es la capital de España. Tiene más de 3 millones de habitantes."

# Definir una pregunta sobre el contexto anterior
pregunta = "¿Cuál es la capital de España?"

# Usar el pipeline para obtener una respuesta
respuesta = qa_model(question=pregunta, context=contexto)

# Mostrar la respuesta
print("Respuesta:", respuesta["answer"])


Device set to use cpu


Respuesta: Madrid


In [5]:
# Cargar el dataset SQuAD versión 1
squad = load_dataset("squad")

# Ver un ejemplo
print("Contexto:", squad['train'][0]['context'])
print("Pregunta:", squad['train'][0]['question'])
print("Respuesta:", squad['train'][0]['answers']['text'][0])

Contexto: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Pregunta: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Respuesta: Saint Bernadette Soubirous


In [6]:
# Usar el pipeline sobre una muestra real del dataset SQuAD
contexto = squad['train'][0]['context']
pregunta = squad['train'][0]['question']
respuesta_real = squad['train'][0]['answers']['text'][0]

respuesta_modelo = qa_model(question=pregunta, context=contexto)

print("Pregunta:", pregunta)
print("Contexto:", contexto)
print("Respuesta esperada:", respuesta_real)
print("Respuesta del modelo:", respuesta_modelo["answer"])


Pregunta: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Contexto: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Respuesta esperada: Saint Bernadette Soubirous
Respuesta del modelo: Saint Bernadette Soubirous


## Cargar el modelo y el tokenizer fine-tuned

In [15]:

# Cargar el tokenizer y modelo base (no entrenado para QA todavía)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-cased-distilled-squad')
model = AutoModelForQuestionAnswering.from_pretrained('distilbert-base-cased-distilled-squad')

# Función para tokenizar cada ejemplo
def preprocess(example):
    return tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

# Aplicar tokenización
tokenized_squad = squad.map(preprocess, batched=True, remove_columns=squad["train"].column_names)

Map: 100%|██████████| 10570/10570 [00:04<00:00, 2284.60 examples/s]


In [16]:
def add_labels(example):
    start_positions = []
    end_positions = []

    for i in range(len(example["offset_mapping"])):
        offsets = example["offset_mapping"][i]
        input_ids = example["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Usar -100 como valor padrão se a resposta não for encontrada
        start_pos = cls_index
        end_pos = cls_index

        answer = squad["train"][i]["answers"]["text"][0]
        start_char = squad["train"][i]["answers"]["answer_start"][0]
        end_char = start_char + len(answer)

        for idx, (start, end) in enumerate(offsets):
            if start <= start_char < end:
                start_pos = idx
            if start < end_char <= end:
                end_pos = idx

        start_positions.append(start_pos)
        end_positions.append(end_pos)

    example["start_positions"] = start_positions
    example["end_positions"] = end_positions
    return example

# Aplicar los labels
tokenized_squad = tokenized_squad.map(add_labels, batched=True)

Map: 100%|██████████| 10822/10822 [00:12<00:00, 863.89 examples/s]


## Evaluación del modelo con la métrica oficial de SQuAD (Exact Match y F1)

In [ ]:
# Cargar la métrica oficial de SQuAD para exact match y F1
squad_metric = evaluate.load("squad")

# Crear una función para predecir una respuesta a partir de pregunta y contexto
def get_answer(question, context):
    inputs = tokenizer(question, context, return_tensors="pt")
    outputs = model(**inputs)

    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end])
    )
    return answer.strip()

# Ejemplo de prueba (puedes cambiar por cualquier otro)
context = "Madrid es la capital de España."
question = "¿Cuál es la capital de España?"
predicted_answer = get_answer(question, context)

# Preparar los datos para la métrica (formato oficial)
predictions = [{"id": "0", "prediction_text": predicted_answer}]
references = [{"id": "0", "answers": {"answer_start": [11], "text": ["Madrid"]}}]

# Calcular las métricas Exact Match y F1
results = squad_metric.compute(predictions=predictions, references=references)

# Mostrar los resultados
print(f"Predicción del modelo: {predicted_answer}")
print(f"Métricas -> Exact Match: {results['exact_match']}%, F1: {results['f1']}%")


Predicción del modelo: Madrid
Métricas -> Exact Match: 100.0%, F1: 100.0%


In [9]:
# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./bert-qa-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs"
)


# Crear el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["validation"],
)

# Empezar el entrenamiento
trainer.train()


# Guardar el modelo y tokenizer entrenado
model.save_pretrained("./modelo_finetuned_qa")
tokenizer.save_pretrained("./modelo_finetuned_qa")


c:\Users\nunoc\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,4.182800


KeyboardInterrupt: 

## Conclusión

En este trabajo se ha abordado la tarea de **Pregunta-Respuesta (Question Answering)** mediante el uso de modelos **Transformers** y la librería Hugging Face. El propósito fue demostrar cómo un modelo preentrenado puede aplicarse de forma directa para responder preguntas a partir de un contexto, utilizando el conjunto de datos **SQuAD 1.0** como referencia.

Inicialmente se intentó realizar el **fine-tuning del modelo `bert-base-uncased`** para ajustar sus pesos a la tarea específica. Sin embargo, debido a limitaciones de hardware, ya que se utilizó una máquina local sin GPU, el proceso estimaba un tiempo superior a **36 horas**. Por este motivo, se decidió interrumpir dicho entrenamiento.

Como alternativa, se empleó el modelo **`distilbert-base-cased-distilled-squad`**, el cual ya ha sido previamente fine-tuned para esta tarea concreta. A través del pipeline de Hugging Face, se realizaron pruebas tanto con ejemplos simples creados manualmente como con ejemplos reales del dataset SQuAD. En todos los casos, el modelo fue capaz de extraer la respuesta correcta, obteniendo un **100% en las métricas Exact Match y F1**.